# Segmentation fine-tuning on Kvasir-SEG

SegFormer all-MLP decoder on a ViT simple feature pyramid, 352px, 100 epochs.
Five encoders x five seeds = 25 runs, ~4 GPU-hours total.

Every arm uses the **identical** decoder, recipe, splits and seeds — only the encoder weights differ. Test is scored exactly once, on the best-val checkpoint.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    """Run a shell command, streaming its output live.

    Streaming rather than capture_output matters here: the corpus resize runs
    for 20-40 minutes and prints a progress line with an ETA every 256 images.
    Buffering that until the process exits makes a long job indistinguishable
    from a hung one.
    """
    print('$', cmd, flush=True)
    # PYTHONUNBUFFERED: a child process writing to a pipe switches from line
    # buffering to 4 KB block buffering, so progress lines would still arrive
    # in bursts (or not at all until exit) even though we stream them here.
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code_ = p.wait()
    if check and code_ != 0:
        # Include the tail of the output in the exception. Otherwise the
        # traceback shows only this wrapper and the real error is buried
        # further up the cell, which is easy to miss and impossible to
        # copy/paste usefully.
        tail = ''.join(lines[-25:]).rstrip()
        raise RuntimeError(
            f'command failed (exit {code_}): {cmd}\n\n--- last output ---\n{tail}')
    return subprocess.CompletedProcess(cmd, code_, ''.join(lines), '')

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


In [ ]:
# Pull the exported encoders published by the pretraining notebooks.
# Recursive: Kaggle nests mounts as /kaggle/input/datasets/<owner>/<slug>/,
# so a fixed path finds nothing and segmentation then reports the confusing
# 'run pretraining first' for a run that already finished.
import shutil, glob, pathlib
dst = pathlib.Path('/kaggle/working/weights')
dst.mkdir(parents=True, exist_ok=True)
found = [p for p in glob.glob('/kaggle/input/**/*.pt', recursive=True)
         if 'jepa-thesis-weights' in p]
for p in found:
    shutil.copy(p, dst)
print('encoders available:', sorted(f.name for f in dst.glob('*.pt')) or 'NONE')
if not found:
    print('Attach the jepa-thesis-weights dataset via + Add Input. '
          '(Only the `random` arm can run without it.)')


In [ ]:
ENCODERS = ['ijepa', 'mae', 'simclr', 'mocov3', 'random']
SEEDS = [0, 1, 2, 3, 4]
for enc in ENCODERS:
    for seed in SEEDS:
        sh(f'python -m src.engine.segment --encoder {enc} --seed {seed}', check=False)


## Ablations: low-label regime and decoder swap

In [ ]:
for enc in ENCODERS:
    for lf in [0.1, 0.25, 0.5]:
        for seed in [0, 1, 2]:
            sh(f'python -m src.engine.segment --encoder {enc} --label-fraction {lf} --seed {seed}', check=False)


In [ ]:
for enc in ENCODERS:
    sh(f'python -m src.engine.segment --encoder {enc} --decoder unet --seed 0', check=False)


In [ ]:
# Comparability table against published Kvasir-SEG numbers.
for enc in ENCODERS:
    sh(f'python -m src.engine.segment --encoder {enc} --split 880_120 --seed 0', check=False)
